# Oscilloscope acquisition

Works with **Tektronix** and **Agilent/Keysight** scopes.  
Brand is detected automatically; all SCPI differences are handled in `utils/oscilloscope.py`.

If a new oscilloscope from a different brand is added, one just needs to add the correct syntax and if/else statements in the utils file.

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyvisa as pv

import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from utils.naming import make_run_dir

Utils import for oscilloscope readout

In [ ]:
from utils.oscilloscope import (
    get_brand,
    get_active_channels,
    acquire_waveform,
    stop_acquisition,
    run_acquisition,
    save_screenshot,
)

**OPO**

In [ ]:
# FOLDER
SAMPLE = "opo_cavity_alignement_seed_lock_empty"
MEAS = "oscilloscope_trace"
ACQ_CELL = f"{MEAS}_{SAMPLE}"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

print("ACQ_CELL:", ACQ_CELL)
print("Oscilloscope data will be saved in:", RUN_DIR)

## 1. Connect to oscilloscope

Run `rm.list_resources()` to find the right VISA address, then paste it below. 
Run one of the two scope cells below to select the correct oscilloscope.

In [ ]:
rm = pv.ResourceManager()
print(rm.list_resources())

*Tektronik*

In [ ]:
visa_adress_tektronik = 'USB0::0x0699::0x039F::C010359::INSTR'
scope = rm.open_resource(visa_adress_tektronik) # Oscilloscope
scope.timeout = 20000  # ms, image transfer can take a moment

*Agilent*

In [ ]:
visa_adress_agilent = 'USB0::0x0957::0x17A6::MY51250269::INSTR'
scope = rm.open_resource(visa_adress_agilent) # Oscilloscope
scope.timeout = 20000  # ms, image transfer can take a moment
scope.chunk_size = 1024 * 1024 # bytes — needed for long Agilent waveforms

**Check connection and name**

In [ ]:
brand = get_brand(scope)       # auto-detect: "tektronix" or "agilent"

print("IDN   :", scope.query("*IDN?").strip())
print("Brand :", brand)

## 2. Experiment folder

In [ ]:
DATA_DIR = "data/oscilloscope"
os.makedirs(DATA_DIR, exist_ok=True)

**SHG**

In [ ]:
# FOLDER
SAMPLE = "shg_cavity_luke"
MEAS = f"{brand}_trace"
ACQ_CELL = f"{MEAS}_{SAMPLE}"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

print("Oscilloscope data will be saved in:", RUN_DIR)

## 3. Acquire all active traces

In [ ]:
active_channels = get_active_channels(scope, brand)
print("Active channels:", active_channels)

In [ ]:
aqm  = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

data = {}  # will hold all waveform arrays, keyed as CH1_x, CH1_y, …
data["brand"] = brand # store brand for later use in analysis

stop_acquisition(scope, brand)   # freeze display for a stable readout

for channel in active_channels:
    wf = acquire_waveform(scope, brand, channel)  # returns dict with x, y, metadata

    # Store with channel prefix so keys never collide across channels
    for key, value in wf.items():
        data[f"{channel}_{key}"] = value

    print(f"  {channel}: {len(wf['x'])} points, "
          f"dt = {wf['xincr']*1e9:.2f} ns, "
          f"V range = [{wf['y'].min():.3f}, {wf['y'].max():.3f}] V")

aqm.save_acquisition(**data)     # save everything to disk at once
run_acquisition(scope, brand)    # unfreeze

print("\nData saved to:", RUN_DIR)

## 4. Quick plot

In [ ]:
for channel in active_channels:
    x = data[f"{channel}_x"]
    y = data[f"{channel}_y"]

    plt.figure(figsize=(8, 3))
    plt.plot(x * 1e3, y)                  # convert s → ms for readability
    plt.title(f"{SAMPLE} — {channel}")
    plt.xlabel("Time (ms)")
    plt.ylabel("Voltage (V)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 5. Screenshot

In [ ]:
SCREENSHOT_DIR = Path(RUN_DIR) / "screenshots"
SCREENSHOT_DIR.mkdir(exist_ok=True)

In [ ]:
# Auto-increment filename so we never overwrite an existing screenshot
filename = SCREENSHOT_DIR / f"{SAMPLE}.png"
index = 1
while filename.exists():
    filename = SCREENSHOT_DIR / f"{SAMPLE}_{index}.png"
    index += 1

save_screenshot(scope, brand, filename)